# 02 – Exploratory Data Analysis (EDA) of Banking77

**Purpose:** Run exploratory data analysis on the preprocessed Banking77 dataset splits, review class frequencies, query lengths, and save summary metrics, tables, and visualization plots.

This notebook demonstrates:
1. Loading preprocessed cached dataset splits.
2. Generating descriptive statistics for text sequence length.
3. Reviewing class label / intent distributions (identifying imbalance or rare classes).
4. Reviewing vocabulary frequencies and top word counts.
5. Saving artifacts to `outputs/eda/` for downstream reporting.

## 0. Environment Setup

Import essential libraries and add repository root to `sys.path`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as display

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Load Preprocessed Data Splits

Load train, val, and test splits from the cache using the data layer coordinator function.

In [ ]:
from src.data.dataset import load_and_preprocess_dataset

splits = load_and_preprocess_dataset()
train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

print(f"Train records: {len(train_df)}")
print(f"Val records:   {len(val_df)}")
print(f"Test records:  {len(test_df)}")

## 2. Trigger EDA Generator Programmatically

We can trigger the EDA artifact generator to output standard plots, tables, and JSON reports under `outputs/eda/`.

In [ ]:
from src.data.eda import generate_eda_artifacts

summary = generate_eda_artifacts()
print("EDA generation complete! Key statistics:")
print(f"Total dataset rows: {summary['total_rows']}")
print(f"Vocabulary size:    {summary['vocabulary_size']} words")
print(f"Unique intents:     {summary['unique_classes']}")
print(f"Rare intents (<100): {summary['rare_classes_count']}")

## 3. Display Visualization Plots

Let's render the generated figures directly in the notebook.

### A. Sentence Length Histogram

In [ ]:
display.Image(filename=str(REPO_ROOT / "outputs" / "eda" / "plots" / "sentence_lengths.png"))

### B. Top 25 Intent Distribution

In [ ]:
display.Image(filename=str(REPO_ROOT / "outputs" / "eda" / "plots" / "intent_distribution.png"))

### C. Top 20 Most Frequent Words

In [ ]:
display.Image(filename=str(REPO_ROOT / "outputs" / "eda" / "plots" / "top_words.png"))

## 4. Review Tabular Summaries

Load intent frequency tables and rare classes.

In [ ]:
freq_df = pd.read_csv(REPO_ROOT / "outputs" / "eda" / "tables" / "intent_frequencies.csv")
print("=== Top 10 Intent Frequencies ===")
print(freq_df.head(10))

In [ ]:
rare_df = pd.read_csv(REPO_ROOT / "outputs" / "eda" / "tables" / "rare_intents.csv")
print(f"\n=== Rare Intents (Total Samples < 100, Count={len(rare_df)}) ===")
print(rare_df.head(15))

In [ ]:
# Export Phase Manifest
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

manifest = {
    "phase": "02_EDA_Banking77",
    "total_rows": summary["total_rows"],
    "train_rows": summary["train_rows"],
    "val_rows": summary["val_rows"],
    "test_rows": summary["test_rows"],
    "unique_classes": summary["unique_classes"],
    "rare_classes_count": summary["rare_classes_count"],
    "vocabulary_size": summary["vocabulary_size"],
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_02_eda.yaml")
print("YAML manifest saved successfully:")
print(manifest)
